# Ablation: agent_v3 without the decision policy

agent_v3 with the model deciding every case instead of the EAU policy.

**Notebook settings:** GPU T4 x2, Internet on, the private dataset `vladyslavsh0/chimera-agent-challenge-task-1` added as input, and the Kaggle secret `HF_TOKEN` (read token with access to `google/embeddinggemma-300m`) attached.

Set `COMMIT` and `SPLIT` below. `SYSTEMS` may list several systems; they then run one after another in this session, which saves the setup time but adds up their run times.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!find /kaggle/input -maxdepth 6 -type d -name agent_input

Dataset path, commit, split, sampling temperature and run name.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

DATA_ROOT = "/kaggle/input/datasets/vladyslavsh0/chimera-agent-challenge-task-1"
COMMIT = ""
SYSTEMS = ["agent_v3_model_decides"]
SPLIT = "test"
TEMPERATURE = 1.0
SUFFIX = f"_{SPLIT}" + ("" if TEMPERATURE == 1.0 else f"_t{TEMPERATURE:g}")
REPO = "/kaggle/working/CHIMERA-Agent"
assert COMMIT and SPLIT in ("dev", "test"), "set COMMIT to the pushed commit and SPLIT to dev or test"
print([s + SUFFIX for s in SYSTEMS])
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
![ -d {REPO} ] || git clone -q https://github.com/VlaDisLav23232/CHIMERA-Agent.git {REPO}
%cd {REPO}
!git checkout -q {COMMIT} && git log -1 --format='%H %s'

Python 3.12 environment, the pinned baseline and the agent dependencies.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 .venv
!uv pip install -q --python .venv/bin/python -r requirements.txt
!bash scripts/setup_agent.sh

Ollama with the context length the runner checks, then the models (Gemma via Ollama, EmbeddingGemma via Hugging Face).

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
ollama = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_CONTEXT_LENGTH": "16384"},
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama --version

In [ ]:
!bash scripts/setup_agent.sh --models
!ollama list

Unit tests on Linux. The evaluation tests need the organizers' evaluator; tests that need local data skip here.

In [ ]:
!bash scripts/fetch_evaluator.sh
!.venv/bin/python -m unittest discover -s tests -t .

A config copy that points `data_dir` at the dataset's agent inputs and sets the temperature. Everything else stays as in `configs/agent.json`. The check after it counts how many cases of the split the dataset has.

In [ ]:
import json
config = json.load(open("configs/agent.json"))
config["data_dir"] = f"{DATA_ROOT}/task1/agent_input"
config["temperature"] = TEMPERATURE
json.dump(config, open("/kaggle/working/agent_kaggle.json", "w"), indent=2)
print(config)
split_ids = json.load(open("splits/task1.json"))[SPLIT]
print(SPLIT, "cases in the dataset:", sum(os.path.isdir(f"{config['data_dir']}/{c}") for c in split_ids), "of", len(split_ids))

Smoke test of each system on the first dev case (about 2 min each): stop here unless every line says `succeeded`.

In [ ]:
first = sorted(json.load(open("splits/task1.json"))["dev"])[0]
for SYSTEM in SYSTEMS:
    !.venv/bin/python -m chimera_agent.run --name {SYSTEM}{SUFFIX}_smoke --system {SYSTEM} --case {first} --config /kaggle/working/agent_kaggle.json
    print(SYSTEM, json.load(open(f"results/{SYSTEM}{SUFFIX}_smoke/run_info.json"))["cases"][first]["status"])

Full run of the split. The three documents are fetched without an LLM call, so a case should take less than the ~100 s of agent_v1/v2; the first finished case in `results/{RUN}/run_info.json` shows the real time.

In [ ]:
for SYSTEM in SYSTEMS:
    !.venv/bin/python -m chimera_agent.run --name {SYSTEM}{SUFFIX} --system {SYSTEM} --all-{SPLIT} --config /kaggle/working/agent_kaggle.json

Run summary: counts only, no patient text.

In [ ]:
import collections, glob
for SYSTEM in SYSTEMS:
    RUN = SYSTEM + SUFFIX
    print("==", RUN)
    info = json.load(open(f"results/{RUN}/run_info.json"))
    print(info["status"], collections.Counter(r["status"] for r in info["cases"].values()))
    print({c[-6:]: (r["status"], r.get("error_type")) for c, r in info["cases"].items() if r["status"] != "succeeded"})
    traces = [json.load(open(p)) for p in glob.glob(f"results/{RUN}/trace/*.json")]
    print("escape repairs:", sum(t["escape_repairs"] for t in traces), "in", sum(t["escape_repairs"] > 0 for t in traces), "cases")
    print("gate:", collections.Counter("none" if t["gate"] is None else t["gate"]["biopsy_decision"] for t in traces))
    !.venv/bin/python -m evaluation.audit checks results/{RUN}

Zip a run without its index copy and download it right away (`/kaggle/working` is lost when the session ends). The zip contains patient information: keep it local, never commit or share it.

In [ ]:
for SYSTEM in SYSTEMS:
    RUN = SYSTEM + SUFFIX
    !cd results && zip -qr /kaggle/working/{RUN}.zip {RUN} {RUN}_smoke -x "{RUN}/resources/*" "{RUN}_smoke/resources/*" && ls -lh /kaggle/working/{RUN}.zip